### **Code : 보고서 작성을 위한 군집분석**
#### Writer : Donghyeon Kim
#### Update : 2026.01.15.

---

#### **0. Prior Settings**

In [1]:
# Library
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 경로 및 상위-상위 경로 설정
folder_root = Path(os.getcwd())
parent_root = folder_root.parent.parent

---

#### **1. Clustering**

In [2]:
# Data 경로
data_path = os.path.join(parent_root, '3. 보고서')

# Rawdata 파일명 및 경로
data_file_name = os.path.join(data_path, '1_TimeCost_본설문_Rawdata_통합_분석용(41개 outlier 필터된).xlsx')
data_file_sheet = 'Sheet1'

# Data 호출
df = pd.read_excel(data_file_name, sheet_name=data_file_sheet)

In [3]:
# 결과물 저장 경로
result_path = os.path.join(parent_root, '3. 보고서', 'Clustering Result')
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [4]:
# 보고서용 군집분석: P2-Q1, 총사용가치, 선택가치 기준으로 3개 군집을 만든다.
CLUSTER_COUNT = 3
OUTPUT_FILE = "본설문_Cluster3.xlsx"  # 보고서에서 사전 명시한 3개 군집 결과
out_path = os.path.join(result_path, OUTPUT_FILE)

CLUSTER_COLUMN_RULES = {
    "P2-Q1": {
        "candidates": ["P2-Q1"],
        "must_contain": ["P2", "Q1"],
    },
    "총사용가치": {
        "candidates": ["총사용가치(원/년)(AG)", "총사용가치(원/년)", "총사용가치"],
        "must_contain": ["총사용가치"],
    },
    "선택가치": {
        "candidates": ["선택가치(원/년)(AH)", "선택가치(원/년)", "선택가치"],
        "must_contain": ["선택가치"],
    },
}


def normalize_col_name(value: str) -> str:
    """컬럼명 비교를 위해 공백과 괄호 표기 차이 줄이기"""
    value = str(value).strip().replace("（", "(").replace("）", ")")
    return re.sub(r"\s+", "", value)


def find_col(cols, candidates=None, must_contain=None):
    """후보명 완전일치, 후보명 포함, 필수 키워드 포함 순서로 컬럼 찾기"""
    normalized = {col: normalize_col_name(col) for col in cols}

    if candidates:
        candidate_keys = [normalize_col_name(candidate) for candidate in candidates]
        for col, col_key in normalized.items():
            if col_key in candidate_keys:
                return col
        for col, col_key in normalized.items():
            if any(candidate_key in col_key for candidate_key in candidate_keys):
                return col

    if must_contain:
        required_keys = [normalize_col_name(keyword) for keyword in must_contain]
        for col, col_key in normalized.items():
            if all(keyword in col_key for keyword in required_keys):
                return col

    return None


def resolve_cluster_cols(df: pd.DataFrame) -> dict:
    """군집분석에 필요한 원자료 컬럼명을 자동으로 찾고, 누락 시 바로 원인 표시"""
    cols = list(df.columns)
    resolved = {
        name: find_col(cols, **rule)
        for name, rule in CLUSTER_COLUMN_RULES.items()
    }
    missing = [name for name, col in resolved.items() if col is None]
    if missing:
        raise ValueError(f"군집 변수 컬럼을 찾지 못함: {missing}\n(참고) 현재 컬럼 일부: {cols[:50]}")
    return resolved


def to_money_num(series: pd.Series) -> pd.Series:
    """'300,000원'처럼 텍스트가 섞인 금액 응답을 숫자로 변환"""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    text = series.astype(str).str.strip()
    text = text.str.replace(",", "", regex=False).str.replace(r"[^0-9\.\-]", "", regex=True)
    text = text.replace({"": np.nan, "-": np.nan, ".": np.nan, "-.": np.nan}).infer_objects(copy=False)
    return pd.to_numeric(text, errors="coerce")


def to_p2q1_numeric_or_code(series: pd.Series) -> pd.Series:
    """P2-Q1은 숫자 추출을 우선 적용하고, 숫자가 거의 없으면 범주 코드로 대체"""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    text = series.astype(str).str.strip()
    num = pd.to_numeric(text.str.extract(r"(-?\d+(?:\.\d+)?)", expand=False), errors="coerce")
    if num.notna().mean() >= 0.30:
        return num

    category = text.replace({"": np.nan, "nan": np.nan, "None": np.nan}).astype("category")
    return pd.to_numeric(pd.Series(category.cat.codes, index=series.index).replace({-1: np.nan}), errors="coerce")


def build_cluster_matrix(df: pd.DataFrame, cluster_cols: dict):
    """분석 변수 3개를 숫자형으로 정리하고, 세 값이 모두 있는 행만 군집에 사용"""
    p2q1_col = cluster_cols["P2-Q1"]
    ag_col = cluster_cols["총사용가치"]
    ah_col = cluster_cols["선택가치"]

    xraw = df[[p2q1_col, ag_col, ah_col]].copy()
    xraw[p2q1_col] = to_p2q1_numeric_or_code(xraw[p2q1_col])
    xraw[ag_col] = to_money_num(xraw[ag_col])
    xraw[ah_col] = to_money_num(xraw[ah_col])

    mask = np.isfinite(xraw).all(axis=1)
    X = xraw.loc[mask].copy()
    if len(X) < CLUSTER_COUNT:
        nn = xraw.notna().sum()
        raise ValueError(
            f"군집을 만들기엔 유효 데이터가 너무 적음. (유효 행 수={len(X)})\n"
            f"- P2-Q1 숫자/코드 변환 후 non-null: {int(nn[p2q1_col])}\n"
            f"- 총사용가치 non-null: {int(nn[ag_col])}\n"
            f"- 선택가치 non-null: {int(nn[ah_col])}\n"
            f"→ 셋 다 동시에 채워진 행이 거의/전혀 없어서 mask가 0이 됨"
        )
    return X, mask


def ordered_kmeans_labels(X: pd.DataFrame, ag_col: str) -> np.ndarray:
    """KMeans 결과를 총사용가치 평균이 낮은 순서대로 1, 2, 3 라벨로 변경"""
    scaled = StandardScaler().fit_transform(X.values)
    labels0 = KMeans(n_clusters=CLUSTER_COUNT, random_state=42, n_init=10).fit_predict(scaled)

    label_order = (
        pd.DataFrame({"label0": labels0, "AG": X[ag_col].values})
        .groupby("label0")["AG"]
        .mean()
        .sort_values()
        .index
        .tolist()
    )
    label_map = {label: i + 1 for i, label in enumerate(label_order)}
    return pd.Series(labels0).map(label_map).astype(int).values

In [5]:
# 1) 출력 파일 경로 설정 (완료)

# 2) 군집 변수 컬럼 자동 탐색
cluster_cols = resolve_cluster_cols(df)
COL_P2Q1 = cluster_cols["P2-Q1"]
COL_AG = cluster_cols["총사용가치"]
COL_AH = cluster_cols["선택가치"]
use_cols = [COL_P2Q1, COL_AG, COL_AH]
print("[확인] 군집 변수로 잡힌 컬럼명:", use_cols)

# 3) 숫자 변환 및 유효 행 추출
X, mask = build_cluster_matrix(df, cluster_cols)

# 4) 표준화 + KMeans 군집화 (완료)

# 5) 총사용가치 평균 기준으로 군집 라벨을 1/2/3 순서화
labels123 = ordered_kmeans_labels(X, COL_AG)

# 6) 원본 데이터 마지막 열에 Cluster 추가 (완료)

# 7) 엑셀 저장 및 요약 출력
df_out = df.copy()
df_out["Cluster"] = np.nan
df_out.loc[mask, "Cluster"] = labels123
df_out["Cluster"] = df_out["Cluster"].astype("Int64")
df_out.to_excel(out_path, index=False)

print(f"[저장 완료] {out_path}")
print("\n[군집별 n]")
print(df_out["Cluster"].value_counts(dropna=False).sort_index())
print("\n[군집별 평균 (금액 변수만)]")
print(df_out.groupby("Cluster")[[COL_AG, COL_AH]].mean().round(0))

[확인] 군집 변수로 잡힌 컬럼명: ['P2-Q1', '총사용가치(원/년)', '선택가치(원/년)']
[저장 완료] /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/3. 보고서/Clustering Result/본설문_Cluster3.xlsx

[군집별 n]
Cluster
1       64
2       37
3       13
<NA>     3
Name: count, dtype: Int64

[군집별 평균 (금액 변수만)]
         총사용가치(원/년)   선택가치(원/년)
Cluster                        
1          745234.0   2112969.0
2         1491216.0   3351622.0
3        13100000.0  14615385.0
